# **Import Libraries**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix


# **Load Datasets**

In [8]:
rec_df = pd.read_csv('/content/drive/MyDrive/Crop Minor Project/Crop_recommendation.csv')
yield_df = pd.read_csv('/content/drive/MyDrive/Crop Minor Project/crop_yield.csv')

print("Crop Recommendation Columns:", rec_df.columns)
print("Crop Yield Columns:", yield_df.columns)


Crop Recommendation Columns: Index(['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label'], dtype='object')
Crop Yield Columns: Index(['Crop', 'Crop_Year', 'Season', 'State', 'Area', 'Production',
       'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Yield'],
      dtype='object')


In [9]:

# Normalize crop names for matching
rec_df['label_clean'] = rec_df['label'].str.strip().str.lower()
yield_df['Crop_clean'] = yield_df['Crop'].str.strip().str.lower()

# Expanded mapping for more possible matches
mapping = {
    'blackgram': 'urad',
    'chickpea': 'gram',
    'coconut': 'coconut',
    'cotton': 'cotton(lint)',
    'kidneybeans': 'cowpea(lobia)',
    'lentil': 'masoor',
    'maize': 'maize',
    'mothbeans': 'moth',
    'mungbean': 'moong(green gram)',
    'muskmelon': 'other summer pulses',
    'orange': 'other kharif pulses',
    'papaya': 'other summer pulses',
    'pigeonpeas': 'arhar/tur',
    'pomegranate': 'other rabi pulses',
    'rice': 'rice',
    'watermelon': 'other summer pulses',
    'banana': 'banana',
    'apple': 'other summer pulses',        # No direct match
    'grapes': 'other kharif pulses',       # No direct match
    'coffee': 'other kharif pulses',       # No direct match
    'mango': 'other summer pulses'         # No direct match
    # Add more mappings for additional semantic matches if needed
}

# Apply the mapping to the label_clean column
rec_df['label_clean'] = rec_df['label_clean'].replace(mapping)

# Merge the datasets using mapped and normalized columns
merged_df = pd.merge(
    rec_df, yield_df,
    left_on='label_clean', right_on='Crop_clean',
    how='outer', indicator=True
)

# Filter for rows where the crop was present in both datasets
both_df = merged_df[merged_df['_merge'] == 'both']

# Print number of unique matched crops
unique_matched_crops = both_df['label_clean'].unique()
print("Unique matched crops:", unique_matched_crops)
print("Number of unique matched crops:", len(unique_matched_crops))

# Save the matched rows (preview) if desired
both_df.head().to_csv('/content/drive/MyDrive/Crop Minor Project/matched_crops_expanded.csv', index=False)


Unique matched crops: ['arhar/tur' 'banana' 'coconut' 'cotton(lint)' 'cowpea(lobia)' 'gram'
 'jute' 'maize' 'masoor' 'moong(green gram)' 'moth' 'other kharif pulses'
 'other summer pulses' 'rice' 'urad']
Number of unique matched crops: 15


In [10]:
yield_only = merged_df[merged_df['_merge'] == 'right_only']
unique_yield_only_crops = yield_only['Crop_clean'].unique()
print("Yield-only crops:", unique_yield_only_crops)
print("Number of yield-only crops:", len(unique_yield_only_crops))


Yield-only crops: ['arecanut' 'bajra' 'barley' 'black pepper' 'cardamom' 'cashewnut'
 'castor seed' 'coriander' 'dry chillies' 'garlic' 'ginger' 'groundnut'
 'guar seed' 'horse-gram' 'jowar' 'khesari' 'linseed' 'mesta' 'niger seed'
 'oilseeds total' 'onion' 'other  rabi pulses' 'other cereals'
 'other oilseeds' 'peas & beans (pulses)' 'potato' 'ragi'
 'rapeseed &mustard' 'safflower' 'sannhamp' 'sesamum' 'small millets'
 'soyabean' 'sugarcane' 'sunflower' 'sweet potato' 'tapioca' 'tobacco'
 'turmeric' 'wheat']
Number of yield-only crops: 40


In [21]:

# List of new crops ("yield-only") you want to add
yield_only_crops = ['arecanut' 'bajra' 'barley' 'black pepper' 'cardamom' 'cashewnut'
 'castor seed' 'coriander' 'dry chillies' 'garlic' 'ginger' 'groundnut'
 'guar seed' 'horse-gram' 'jowar' 'khesari' 'linseed' 'mesta' 'niger seed'
 'oilseeds total' 'onion' 'other  rabi pulses' 'other cereals'
 'other oilseeds' 'peas & beans (pulses)' 'potato' 'ragi'
 'rapeseed &mustard' 'safflower' 'sannhamp' 'sesamum' 'small millets'
 'soyabean' 'sugarcane' 'sunflower' 'sweet potato' 'tapioca' 'tobacco'
 'turmeric' 'wheat']

# Calculate feature means from existing data for imputation
mean_features = rec_df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']].mean()

# Create imputed rows for each yield-only crop
imputed_rows = []
for crop in yield_only_crops:
    row = mean_features.copy()
    row['label'] = crop
    imputed_rows.append(row)

# Convert to DataFrame and append
imputed_df = pd.DataFrame(imputed_rows)
print(imputed_df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']].isnull().sum())

rec_df_expanded = pd.concat([rec_df, imputed_df], ignore_index=True)

# Save expanded recommendation dataset
rec_df_expanded.to_csv('/content/drive/MyDrive/Crop Minor Project/Crop_recommendation_expanded.csv', index=False)
print("Added imputed entries for yield-only crops and saved to Crop_recommendation_expanded.csv")



N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
dtype: int64
Added imputed entries for yield-only crops and saved to Crop_recommendation_expanded.csv


In [25]:
# Load the expanded recommendation dataset and the yield dataset
rec_df = pd.read_csv('/content/drive/MyDrive/Crop Minor Project/Crop_recommendation_expanded.csv')

# Normalize names
rec_df['label_clean'] = rec_df['label'].str.strip().str.lower()
yield_df['Crop_clean'] = yield_df['Crop'].str.strip().str.lower()

# (Optional but recommended: use mapping from previous guidance for better name matching)
mapping = {
    'blackgram': 'urad',
    'chickpea': 'gram',
    'cotton': 'cotton(lint)',
    'kidneybeans': 'cowpea(lobia)',
    'lentil': 'masoor',
    'maize': 'maize',
    'mothbeans': 'moth',
    'mungbean': 'moong(green gram)',
    'muskmelon': 'other summer pulses',
    'orange': 'other kharif pulses',
    'papaya': 'other summer pulses',
    'pigeonpeas': 'arhar/tur',
    'pomegranate': 'other rabi pulses',
    'rice': 'rice',
    'watermelon': 'other summer pulses',
    'banana': 'banana',
    'apple': 'other summer pulses',          # No direct match, generic
    'grapes': 'other kharif pulses',         # No direct match, generic
    'coffee': 'other kharif pulses',         # No direct match, generic
    'mango': 'other summer pulses',          # No direct match, generic

    # Add direct matches for new yield-only crops (if any appear in future recommendations)
    'arecanut': 'arecanut',
    'bajra': 'bajra',
    'barley': 'barley',
    'black pepper': 'black pepper',
    'cardamom': 'cardamom',
    'cashewnut': 'cashewnut',
    'castor seed': 'castor seed',
    'coriander': 'coriander',
    'dry chillies': 'dry chillies',
    'garlic': 'garlic',
    'ginger': 'ginger',
    'groundnut': 'groundnut',
    'guar seed': 'guar seed',
    'horse-gram': 'horse-gram',
    'jowar': 'jowar',
    'khesari': 'khesari',
    'linseed': 'linseed',
    'mesta': 'mesta',
    'niger seed': 'niger seed',
    'oilseeds total': 'oilseeds total',
    'onion': 'onion',
    'other  rabi pulses': 'other  rabi pulses',
    'other cereals': 'other cereals',
    'other kharif pulses': 'other kharif pulses',
    'other oilseeds': 'other oilseeds',
    'peas & beans (pulses)': 'peas & beans (pulses)',
    'potato': 'potato',
    'ragi': 'ragi',
    'rapeseed &mustard': 'rapeseed &mustard',
    'safflower': 'safflower',
    'sannhamp': 'sannhamp',
    'sesamum': 'sesamum',
    'small millets': 'small millets',
    'soyabean': 'soyabean',
    'sugarcane': 'sugarcane',
    'sunflower': 'sunflower',
    'sweet potato': 'sweet potato',
    'tapioca': 'tapioca',
    'tobacco': 'tobacco',
    'turmeric': 'turmeric',
    'wheat': 'wheat',
    'other summer pulses': 'other summer pulses'
}

rec_df['label_clean'] = rec_df['label_clean'].replace(mapping)

# Merge using normalized names
merged_df = pd.merge(
    rec_df, yield_df,
    left_on='label_clean', right_on='Crop_clean',
    how='outer', indicator=True
)

# Save fully merged dataset
merged_df.to_csv('/content/drive/MyDrive/Crop Minor Project/merged_expanded_dataset.csv', index=False)
missing_env = merged_df[merged_df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']].isnull().any(axis=1)]
missing_env.head()
print("Rows with missing environmental values:", missing_env.shape[0])
print("Expanded and merged dataset saved as 'merged_expanded_dataset.csv'")


Rows with missing environmental values: 13012
Expanded and merged dataset saved as 'merged_expanded_dataset.csv'


In [20]:
df = pd.read_csv('merged_expanded_dataset.csv')
df.head()

,N,P,K,temperature,humidity,ph,rainfall,label,label_clean,Crop,...,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield,Crop_clean,_merge
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Arecanut,...,Whole Year,Assam,73814.0,56708.0,2051.4,7024878.38,22882.34,0.796087,arecanut,right_only
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Arecanut,...,Whole Year,Karnataka,93100.0,133342.0,1266.7,8860327.00,28861.00,1.293571,arecanut,right_only
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Arecanut,...,Whole Year,Kerala,76145.0,93995.0,3252.4,7246719.65,23604.95,1.147857,arecanut,right_only
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Arecanut,...,Whole Year,Meghalaya,9569.0,12116.0,3818.2,910681.73,2966.39,1.245714,arecanut,right_only
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Arecanut,...,Whole Year,West Bengal,8058.0,12423.0,1852.9,766879.86,2497.98,1.691765,arecanut,right_only


## **Data Cleaning**

In [ ]:
# Removing duplicates
df_recommend.drop_duplicates(inplace=True)
df_yield.drop_duplicates(inplace=True)

# Filling missing values with column mean for all numeric columns
df_recommend.fillna(df_recommend.select_dtypes(include=np.number).mean(), inplace=True)
df_yield.fillna(df_yield.select_dtypes(include=np.number).mean(), inplace=True)

NameError: name 'df_recommend' is not defined

# **Merge Datasets Fully on Crop**
# label - crop_recommendation
# Crop - crop_yield

In [ ]:
df_recommend['label_clean'] = df_recommend['label'].str.strip().str.lower()
df_yield['Crop_clean'] = df_yield['Crop'].str.strip().str.lower()

print("Crop recommendation dataset:")
print(sorted(df_recommend['label_clean'].unique()))
print("Crop yield dataset:")
print(sorted(df_yield['Crop_clean'].unique()))




In [ ]:
mapping = {
    'blackgram': 'urad',
    'chickpea': 'gram',
    'cotton': 'cotton(lint)',
    'grapes': 'other kharif pulses',  # No direct match, use generic
    'kidneybeans': 'cowpea(lobia)',
    'lentil': 'masoor',
    'maize': 'maize',
    'mango': 'other summer pulses',   # No direct match, use generic
    'mothbeans': 'moth',
    'mungbean': 'moong(green gram)',
    'muskmelon': 'other rabi pulses',  # No direct match, use generic
    'orange': 'other kharif pulses',   # No direct match, use generic
    'papaya': 'other summer pulses',   # No direct match, use generic
    'pigeonpeas': 'arhar/tur',
    'pomegranate': 'other rabi pulses', # No direct match, use generic
    'rice': 'rice',
    'watermelon': 'other summer pulses' # No direct match, use generic
    # Add additional mappings if any other clear semantic matches are determined
}


In [ ]:
# Apply the mapping to the cleaned label column
df_recommend['label_clean'] = df_recommend['label_clean'].replace(mapping)

In [ ]:
merged = pd.merge(
    df_recommend, df_yield,
    left_on='label_clean', right_on='Crop_clean', how='outer', indicator=True
)

# Fill missing values
num_cols = merged.select_dtypes(include=['number']).columns
cat_cols = merged.select_dtypes(include=['object']).columns

merged[num_cols] = merged[num_cols].fillna(merged[num_cols].mean())
merged[cat_cols] = merged[cat_cols].fillna('unknown')

# NOW: Remove placeholder/unknown rows
merged = merged[merged['label_clean'] != 'unknown']

print("Merged dataset shape:", merged.shape)
print("Unique crops in merged dataset:", merged['label_clean'].nunique())
print("Crop labels:", merged['label_clean'].unique())
merged.head()

In [ ]:
# If you want to view only the matched entries:
both_df = merged[merged['_merge'] == 'both']

# View the first few matched rows
both_df.head()

In [ ]:
unique_matched_crops = merged[merged['_merge'] == 'both']['label_clean'].unique()
print("Unique matched crops:", unique_matched_crops)
print("Count of unique matched crops:", len(unique_matched_crops))


In [ ]:
print(merged['label_clean'].unique())

# **Select relevant feature**


*   Location(state)
*   Season

*   Annual Rainfall
*   Temperature






In [ ]:
# Encode categorical columns
le_state = LabelEncoder()
merged['State_enc'] = le_state.fit_transform(merged['State'])

le_season = LabelEncoder()
merged['Season'] = merged['Season'].str.strip() # Strip whitespace from season names
le_season.fit(merged['Season']) # Re-fit the LabelEncoder on the cleaned season names
merged['Season_enc'] = le_season.transform(merged['Season'])

# Prepare feature matrix (X) and label vector (y)
X = merged[['State_enc', 'Season_enc', 'Annual_Rainfall', 'temperature']]
y = merged['label']

# Encode labels (crops)
le_crop = LabelEncoder()
y = le_crop.fit_transform(y)

# **Data Validation**
Tempearture and Rainfall Ranges

In [ ]:
print("Minimum temperature:", merged['temperature'].min())
print("Maximum temperature:", merged['temperature'].max())
print("Min rainfall:", merged['Annual_Rainfall'].min())
print("Max rainfall:", merged['Annual_Rainfall'].max())

# Also check for NaN or missing values
print("Missing rainfall values:", merged['Annual_Rainfall'].isna().sum())
print("Missing temperature values:", merged['temperature'].isna().sum())


In [ ]:
# Clip rainfall to valid range (300 - 6550 mm)
merged['Annual_Rainfall'] = merged['Annual_Rainfall'].clip(lower=300, upper=6550)
# Clip temperature to valid range (7 - 45 °C)
merged['temperature'] = merged['temperature'].clip(lower=7, upper=45)

In [ ]:
def validate_temperature(temp):
    if not isinstance(temp, (int, float)):
        print("Temperature must be a number.")
        return False
    if temp < 7 or temp > 45:
        print("Temperature must be between 7°C and 45°C.")
        return False
    return True

def validate_rainfall(rain):
    if not isinstance(rain, (int, float)):
        print("Rainfall must be a number.")
        return False
    if rain < 300 or rain > 6550:
        print("Annual rainfall must be between 300mm and 6550mm.")
        return False
    return True

# Apply validation, explicitly converting to numeric and handling potential NaNs from fillna
merged['temperature'] = merged['temperature'].astype(float).apply(validate_temperature)
merged['Annual_Rainfall'] = merged['Annual_Rainfall'].astype(float).apply(validate_rainfall)

# **Split Data Into Train/Test Sets**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# **Train the Random Forest Classifier**

In [ ]:
rf_model = RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)


# **Evaluate Model Performance**

In [ ]:
y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


# **Create the Crop Recommendation Function**
Farmer inputs: location, season, annual rainfall, temperature (validated).

In [ ]:
def recommend_crop(location, season, annual_rainfall, temperature):

    # Run validation checks
    if not validate_temperature(temperature) or not validate_rainfall(annual_rainfall):
        print("Invalid input values. Please check and try again.")
        return  # Stop execution entirely, no prediction

    # Available encoder classes
    state_classes = list(le_state.classes_)
    season_classes = list(le_season.classes_)

    # Handle unrecognized location
    if location not in state_classes:
        print(f"'{location}' is not recognized in the dataset. Please enter a valid state.")
        return  # stop further execution without predicting

    # Handle unrecognized season
    if season not in season_classes:
        print(f"'{season}' is not recognized in the dataset. Please enter a valid season.")
        return  # stop further execution without predicting

    # Encode categorical values
    loc_enc = le_state.transform([location])[0]
    sea_enc = le_season.transform([season])[0]

    # Prepare features and predict
    features = np.array([[loc_enc, sea_enc, annual_rainfall, temperature]])
    prediction = rf_model.predict(features)
    crop_name = le_crop.inverse_transform(prediction)

    return crop_name[0]


In [ ]:
print("Seasons recognized by model:", le_season.classes_)


In [ ]:
print("States recognized by model:", le_state.classes_)


In [ ]:
# Example crop recommendation
print(recommend_crop('Maharashtra', 'Summer', 1400, 30))


In [ ]:
print(recommend_crop('Andhra Pradesh', 'Whole Year', 1500, 29))


In [ ]:
print(recommend_crop('Rajasthan', 'kharif', 400, 36))


In [ ]:
print(recommend_crop('Kerala', 'kharif', 1200, 100))


In [ ]:
print(recommend_crop('Punjab', 'Rabi', 700, 19))
